In [ ]:
#| default_exp execute

## Executing notebooks

Execute notebooks with visible outputs, local package imports, and a fast per-cell timeout. Cells that exceed the timeout are marked with source-hash metadata and skipped on later runs until their source changes.

Execution closes the loop after reading and writing. The project needs a way to run notebooks as notebooks, with local imports available and with visible outputs copied back into the notebook for inspection.

This notebook wraps `execnb` with nbdev-friendly defaults: local import paths, optional partial execution, timeout handling, and a test helper that reports notebook errors in a concise form.

Execution is the confidence step after an edit. The wrapper keeps imports close to how nbdev users run notebooks, records visible outputs for inspection, and marks timed-out cells so repeated runs do not keep blocking on the same long operation.

```python
exec_nb("nbs/03_execute.ipynb", up2id="some-cell-id", timeout=10, show_output=True)
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from fastcore.nbio import read_nb as _read_nb
from nbskill.execute import exec_nb as _example_exec_nb
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook
from nbskill.write import update_cell, write_nb

In [ ]:
path = demo_path("03_execute_example.ipynb")
try:
    _example_write_nb(str(path), "%%code\nvalue = 6 * 7\nprint(value)", replace=True, export=False)
    _example_exec_nb(str(path), timeout=5, show_output=True)
finally:
    remove_demo_path(path)

In [ ]:
#| export
import ast
import asyncio
import base64
import hashlib
import json
import re
import sys
import threading
import traceback
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from io import StringIO
from pathlib import Path

import pyskills
from execnb.shell import CaptureShell
from fastcore.nbio import read_nb as _read_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import call_parse
from safepyrun import RunPython
from safepyrun.core import allow as _safepyrun_allow

from nbskill.foundation import (
    cell_metadata, cli_error, cli_return, one_chapter, parse_literal,
    stamp_notebook_metadata, tracked_call,
)
from nbskill.parallel import execution_slot, notebook_locks

### Choosing how much to run

Sometimes verification only needs the first few cells or a single chapter. These helpers translate an index, a cell id, or a chapter into pre/post hooks that stop execution at the right point.

In [ ]:
#| export
def _exec_limiters(up2id):
    up2id = parse_literal(up2id)
    noop = lambda cell: None
    if up2id is None: return (lambda cell: False), noop
    if isinstance(up2id, int):
        if up2id < 0: raise ValueError("up2id must be >= 0")
        return (lambda cell: cell.idx_ >= up2id), noop

    done = False
    def preproc(cell): return done
    def postproc(cell):
        nonlocal done
        if cell.id == str(up2id): done = True
    return preproc, postproc

### Importing like the project does

Notebook execution should see the same local package that tests and examples see. This section finds the project root from common markers and puts the notebook folder, root, and optional `src` folder on the shell path.

In [ ]:
#| export
def _project_root_for_notebook(path):
    path = Path(path).resolve()
    start = path.parent if path.suffix else path
    markers = ("pyproject.toml", "settings.ini", "nbdev.yml", ".git")
    for folder in (start, *start.parents):
        if any((folder / marker).exists() for marker in markers): return folder
    if start.name in {"nbs", "notebooks"} and start.parent != start.parent.parent: return start.parent
    return start

In [ ]:
#| export
def _local_import_paths(path):
    nb_dir = Path(path).resolve().parent
    root = _project_root_for_notebook(path)
    paths = [nb_dir, root]
    src = root / "src"
    if src.exists(): paths.append(src)
    return [p for i, p in enumerate(paths) if p.exists() and p not in paths[:i]]

In [ ]:
#| export
_SAFE_SENTINEL = "__nbskill_safe_exec__"

_DEFAULT_CACHE_DOMAINS = (
    "chatgpt.com", "api.openai.com", "api.anthropic.com",
    "generativelanguage.googleapis.com", "api.deepseek.com",
    "api.fireworks.ai", "openrouter.ai", "api.groq.com",
    "api.together.xyz", "api.mistral.ai", "api.x.ai", "api.moonshot.ai",
)

_CACHY_NORM_PATS = [
    (re.compile(r"/ipykernel_\d+/\d+\.py"), "/ipykernel_N/X.py"),
    (re.compile(r"<ipython-input-\d+-\w+>"), "<ipython-input>"),
    (re.compile(r"/var/folders/[\w|/]+"), "/tmp/tmpT"),
    (re.compile(r"/tmp/tmp\w+"), "/tmp/tmpX"),
    (re.compile(r"ipykernel_\d+"), "ipykernel_N"),
    (re.compile(r"0x[0-9a-fA-F]{6,}"), "0xMEM"),
    (re.compile(r"line \d+, in"), "line N, in"),
]


def _parse_str_list(value, default=None):
    value = parse_literal(value)
    if value is None: return list(default or [])
    if isinstance(value, (list, tuple, set)): return [str(o) for o in value if str(o).strip()]
    if isinstance(value, str): return [part.strip() for part in value.split(",") if part.strip()]
    return [str(value)]


@contextmanager
def _temporary_sys_path(paths):
    old_path = list(sys.path)
    for path in reversed([str(Path(p)) for p in paths]):
        if path not in sys.path: sys.path.insert(0, path)
    try: yield
    finally: sys.path[:] = old_path


@contextmanager
def _temporary_allow_registry():
    snapshot = {key: set(value) for key, value in pyskills.__pytools__.items()}
    try: yield
    finally:
        pyskills.__pytools__.clear()
        for key, value in snapshot.items(): pyskills.__pytools__[key].update(value)


def _resolve_allowed_name(name, namespace):
    if name in namespace: return namespace[name]
    parts = str(name).split(".")
    for i in range(len(parts), 0, -1):
        module_name = ".".join(parts[:i])
        try:
            module = __import__(module_name, fromlist=["*"])
            obj = module
            for part in parts[i:]: obj = getattr(obj, part)
            return obj
        except (ImportError, AttributeError):
            continue
    raise ValueError(f"Could not resolve allow entry {name!r}")


def _register_allowed(allow, namespace):
    for name in _parse_str_list(allow):
        _safepyrun_allow(_resolve_allowed_name(name, namespace))


def _cachy_content(request):
    if not hasattr(request, "_content"): request.read()
    content_type = request.headers.get("Content-Type", "").encode()
    boundary = None
    try:
        import httpx
        boundary = httpx._multipart.get_multipart_boundary_from_content_type(content_type)
    except Exception:
        boundary = None
    return request.content.replace(boundary, b"cachy-boundary") if boundary else request.content


def _cachy_normalize(data):
    text = data.decode("utf-8", errors="replace")
    for pattern, replacement in _CACHY_NORM_PATS: text = pattern.sub(replacement, text)
    return text.encode("utf-8")


def _cachy_norm_content(request):
    content = _cachy_content(request)
    if "json" in request.headers.get("content-type", "").lower():
        try: return json.dumps(json.loads(content), sort_keys=True).encode()
        except Exception: pass
    return content


def _cachy_key(request, is_stream=False):
    url = request.url.copy_remove_param("key")
    data = f"{url}{bool(is_stream)}".encode() + _cachy_normalize(_cachy_norm_content(request))
    return hashlib.sha256(data).hexdigest()[:8]


def _cache_path_for_notebook(path, cache_dir=None):
    base = Path(cache_dir) if cache_dir else _project_root_for_notebook(path)
    return base / "cachy.jsonl"


def _cached_response(key, cache_path, request):
    if not cache_path.exists(): return None
    import httpx
    with cache_path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip(): continue
            entry = json.loads(line)
            if entry.get("key") != key: continue
            content = entry.get("response", "")
            if entry.get("binary"): content = base64.b64decode(content)
            return httpx.Response(
                status_code=entry.get("status_code", 200),
                content=content,
                headers=entry.get("headers"),
                request=request,
            )
    return None


def _url_allowed(url, domains):
    return any(domain in str(url) for domain in domains)


@contextmanager
def _httpx_guard(path, cache_httpx=False, cache_dir=None, cache_domains=None):
    try: import httpx
    except ImportError:
        yield
        return
    _safepyrun_allow({
        httpx: ["get", "post", "put", "patch", "delete", "head", "options", "request", "stream"],
        httpx.Client: ["send", "request", "get", "post", "put", "patch", "delete", "head", "options", "stream"],
        httpx.AsyncClient: ["send", "request", "get", "post", "put", "patch", "delete", "head", "options", "stream"],
    })
    original_sync, original_async = httpx.Client.send, httpx.AsyncClient.send
    original_funcs = {
        name: getattr(httpx, name)
        for name in ("request", "get", "post", "put", "patch", "delete", "head", "options")
        if hasattr(httpx, name)
    }
    domains = tuple(_parse_str_list(cache_domains, default=_DEFAULT_CACHE_DOMAINS))
    cache_path = _cache_path_for_notebook(path, cache_dir)

    def from_cache(request, is_stream):
        if not cache_httpx:
            raise RuntimeError(f"nbskill safe execution blocked live httpx call to {request.url}")
        if not _url_allowed(request.url, domains):
            raise RuntimeError(f"nbskill safe execution has no cached domain rule for {request.url}")
        key = _cachy_key(request, is_stream=is_stream)
        response = _cached_response(key, cache_path, request)
        if response is None:
            raise RuntimeError(f"nbskill safe execution has no cached httpx response for {request.url} (key={key})")
        return response

    def send(self, request, **kwargs):
        return from_cache(request, kwargs.get("stream", False))

    async def asend(self, request, **kwargs):
        return from_cache(request, kwargs.get("stream", False))

    def request(method, url, **kwargs):
        req = httpx.Request(
            method, url, params=kwargs.get("params"), headers=kwargs.get("headers"),
            content=kwargs.get("content"), data=kwargs.get("data"), json=kwargs.get("json"),
        )
        return from_cache(req, kwargs.get("stream", False))

    def method_request(method):
        return lambda url, **kwargs: request(method, url, **kwargs)

    httpx.Client.send = send
    httpx.AsyncClient.send = asend
    httpx.request = request
    for method in ("get", "post", "put", "patch", "delete", "head", "options"):
        setattr(httpx, method, method_request(method.upper()))
    try: yield
    finally:
        httpx.Client.send = original_sync
        httpx.AsyncClient.send = original_async
        for name, func in original_funcs.items(): setattr(httpx, name, func)


def _run_async(coro):
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(coro)
    box = {}
    def target():
        try: box["result"] = asyncio.run(coro)
        except BaseException as exc: box["exc"] = exc
    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    if "exc" in box: raise box["exc"]
    return box.get("result")


def _stream_output(name, text):
    if not text: return None
    return {"output_type": "stream", "name": name, "text": text}


def _result_output(value):
    return {
        "output_type": "execute_result",
        "execution_count": None,
        "metadata": {},
        "data": {"text/plain": repr(value)},
    }


def _error_output(exc):
    tb = traceback.format_exception(type(exc), exc, exc.__traceback__)
    return {"output_type": "error", "ename": type(exc).__name__, "evalue": str(exc), "traceback": tb}


def _magic_error(source):
    for line in source.splitlines():
        stripped = line.lstrip()
        if stripped.startswith("!") or stripped.startswith("%"):
            return RuntimeError("nbskill safe execution blocks IPython shell escapes and magics")
    return None


class _SafeShell:
    def __init__(self, path, extra_paths=None, allow=None, ok_dests=None, cache_httpx=False, cache_dir=None, cache_domains=None):
        self.path = Path(path)
        self.paths = [*(_local_import_paths(path)), *(extra_paths or [])]
        self.cache_httpx = cache_httpx
        self.cache_dir = cache_dir
        self.cache_domains = cache_domains
        self.safe = True
        self.exc = None
        self.g = {
            _SAFE_SENTINEL: True,
            "__name__": "__main__",
            "__file__": str(self.path),
        }
        with _temporary_sys_path(self.paths):
            _register_allowed(allow, self.g)
        self.runner = RunPython(g=self.g, ok_dests=_parse_str_list(ok_dests))

    def run(self, source, timeout=30, verbose=False):
        self.exc = _magic_error(source)
        if self.exc: return [_error_output(self.exc)]
        out, err = StringIO(), StringIO()
        result = None
        try:
            async def call_runner():
                with _temporary_sys_path(self.paths), _httpx_guard(
                    self.path, cache_httpx=self.cache_httpx, cache_dir=self.cache_dir, cache_domains=self.cache_domains,
                ), redirect_stdout(out), redirect_stderr(err):
                    return await self.runner(source)
            coro = call_runner()
            if timeout and timeout > 0: coro = asyncio.wait_for(coro, timeout=timeout)
            result = _run_async(coro)
        except (asyncio.TimeoutError, TimeoutError):
            self.exc = TimeoutError(f"cell ran longer than {timeout}s")
        except BaseException as exc:
            self.exc = exc
        if verbose:
            if out.getvalue(): print(out.getvalue(), end="")
            if err.getvalue(): print(err.getvalue(), end="", file=sys.stderr)
        outputs = [o for o in (_stream_output("stdout", out.getvalue()), _stream_output("stderr", err.getvalue())) if o]
        if self.exc: outputs.append(_error_output(self.exc))
        elif result is not None: outputs.append(_result_output(result))
        return outputs


def _exec_shell(
    path,
    extra_paths=None,
    safe=True,
    allow=None,
    ok_dests=None,
    cache_httpx=False,
    cache_dir=None,
    cache_domains=None,
):
    if safe:
        return _SafeShell(
            path, extra_paths=extra_paths, allow=allow, ok_dests=ok_dests,
            cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains,
        )
    shell = CaptureShell()
    for pth in reversed([*(_local_import_paths(path)), *(extra_paths or [])]):
        shell.set_path(pth)
    return shell

### Timeouts that do not trap future runs

A timed-out cell can make repeated verification painful. Timeout metadata records the source hash that timed out, skips that exact source on later runs, and automatically clears the mark when the cell changes.

In [ ]:
#| export
_TIMEOUT_HASH_KEY = "nbskill_timeout_hash"

In [ ]:
#| export
_TIMEOUT_SECONDS_KEY = "nbskill_timeout_seconds"

_EXECUTED_HASH_KEY = "nbskill_executed_hash"

In [ ]:
#| export
def _source_hash(source):
    return hashlib.sha256(str(source).encode("utf-8")).hexdigest()

In [ ]:
#| export
# _cell_metadata is imported from nbskill.foundation.

In [ ]:
#| export
def _cell_source_hash(cell): return _source_hash(cell.get("source", ""))


class _ExecutionApprovalRequired(RuntimeError): pass


def _cell_has_execution_approval(cell):
    if getattr(cell, "execution_count", None) not in (None, 0): return True
    return cell_metadata(cell).get(_EXECUTED_HASH_KEY) == _cell_source_hash(cell)


def _approval_required_error(cell):
    msg = (
        f"nbskill: refusing to execute unapproved cell id={cell.id}. "
        "Run it once yourself, or rerun nbskill with allow_new=True if you approve this source."
    )
    return _ExecutionApprovalRequired(msg)


def _mark_executed(cell):
    if cell.cell_type == "code": cell_metadata(cell)[_EXECUTED_HASH_KEY] = _cell_source_hash(cell)

In [ ]:
#| export
def _timeout_stream(text):
    return {"output_type": "stream", "name": "stderr", "text": text if text.endswith("\n") else text + "\n"}

In [ ]:
#| export
def _timeout_error(ename, text):
    return {"output_type": "error", "ename": ename, "evalue": text, "traceback": [text]}

In [ ]:
#| export
def _skip_timed_out_cell(cell):
    if cell.cell_type != "code": return False
    meta = cell_metadata(cell)
    current_hash = _cell_source_hash(cell)
    timeout_hash = meta.get(_TIMEOUT_HASH_KEY)
    if timeout_hash == current_hash:
        seconds = meta.get(_TIMEOUT_SECONDS_KEY, "?")
        msg = (
            f"nbskill: skipped cell id={cell.id}; it previously exceeded the "
            f"{seconds}s timeout. Edit the cell to change its hash and rerun it."
        )
        cell.outputs = [_timeout_error("NbskillTimeoutSkipped", msg)]
        return True
    if timeout_hash and timeout_hash != current_hash:
        meta.pop(_TIMEOUT_HASH_KEY, None)
        meta.pop(_TIMEOUT_SECONDS_KEY, None)
    return False

In [ ]:
#| export
def _mark_timeout(cell, timeout, outputs):
    meta = cell_metadata(cell)
    meta[_TIMEOUT_HASH_KEY] = _cell_source_hash(cell)
    meta[_TIMEOUT_SECONDS_KEY] = timeout
    msg = f"nbskill: cell id={cell.id} ran longer than {timeout}s and was stopped."
    cell.outputs = [_timeout_stream(msg), *(outputs or [])]

In [ ]:
#| export
def _clear_timeout_mark(cell):
    meta = cell_metadata(cell)
    meta.pop(_TIMEOUT_HASH_KEY, None)
    meta.pop(_TIMEOUT_SECONDS_KEY, None)

In [ ]:
#| export
def _run_cell(shell, cell, timeout=30, verbose=False, allow_new=False):
    if cell.cell_type != "code": return
    shell._cell_idx = cell.idx_ + 1
    if getattr(shell, "safe", False) and not allow_new and not _cell_has_execution_approval(cell):
        shell.exc = _approval_required_error(cell)
        cell.outputs = [_error_output(shell.exc)]
        return
    outputs = shell.run(cell.source, timeout=timeout if timeout and timeout > 0 else None, verbose=verbose)
    cell.outputs = outputs or []
    if isinstance(shell.exc, TimeoutError): _mark_timeout(cell, timeout, outputs)
    else:
        _clear_timeout_mark(cell)
        if shell.exc is None: _mark_executed(cell)

In [ ]:
#| export
def _execute_nb(
    path,
    dest=None,
    exc_stop=False,
    preproc=lambda cell: False,
    postproc=lambda cell: None,
    timeout=30,
    verbose=False,
    safe=True,
    allow=None,
    ok_dests=None,
    cache_httpx=False,
    cache_dir=None,
    cache_domains=None,
    allow_new=False,
):
    with notebook_locks(path, dest):
        with execution_slot():
            nb = _read_nb(path)
            first_exc = None
            with _temporary_allow_registry():
                shell = _exec_shell(
                    path, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx,
                    cache_dir=cache_dir, cache_domains=cache_domains,
                )
                for cell in nb.cells:
                    if preproc(cell): continue
                    if _skip_timed_out_cell(cell):
                        postproc(cell)
                        continue
                    _run_cell(shell, cell, timeout=timeout, verbose=verbose, allow_new=allow_new)
                    postproc(cell)
                    if isinstance(shell.exc, _ExecutionApprovalRequired):
                        break
                    if shell.exc and exc_stop:
                        first_exc = shell.exc
                        break
            if dest:
                stamp_notebook_metadata(nb)
                _write_nb(nb, dest)
            if first_exc: raise first_exc
            return nb

In [ ]:
#| export
def _text_output(value):
    if value is None: return ""
    if isinstance(value, list): return "".join(map(str, value))
    return str(value)


def _output_text(output):
    otype = output.get("output_type")
    if otype == "stream": return _text_output(output.get("text"))
    if otype == "error":
        tb = output.get("traceback")
        if tb: return _text_output(tb)
        return f"{output.get('ename', 'Error')}: {output.get('evalue', '')}"
    if otype in {"execute_result", "display_data"}:
        data = output.get("data", {})
        for mime in ("text/plain", "text/markdown", "text/html"):
            if mime in data: return _text_output(data[mime])
    return ""


def _is_rich_traceback_stream(output):
    if output.get("output_type") != "stream": return False
    text = _text_output(output.get("text"))
    return "Traceback" in text and "\x1b[" in text


def _executed_cells(nb, up2id=None):
    up2id = parse_literal(up2id)
    if up2id is None: return list(enumerate(nb.cells))
    if isinstance(up2id, int): return list(enumerate(nb.cells[:up2id]))
    items = []
    for idx, cell in enumerate(nb.cells):
        items.append((idx, cell))
        if cell.id == str(up2id): break
    return items


def _print_outputs_from_nb(nb, up2id=None):
    for idx, cell in _executed_cells(nb, up2id):
        outputs = getattr(cell, "outputs", None) or []
        has_error = any(output.get("output_type") == "error" for output in outputs)
        for output in outputs:
            if has_error and _is_rich_traceback_stream(output): continue
            text = _output_text(output)
            if not text: continue
            print(f"--- output id={cell.id} ---")
            print(text, end="" if text.endswith("\n") else "\n")


def _print_nb_outputs(path, up2id=None):
    with notebook_locks(path):
        _print_outputs_from_nb(_read_nb(path), up2id=up2id)

In [ ]:
#| export
def _parse_executed_code_cell(cell):
    if getattr(cell, "cell_type", None) != "code": return None
    try: return ast.parse(getattr(cell, "source", ""))
    except SyntaxError: return None

In [ ]:
#| export
def _top_level_call_names(tree):
    calls = set()

    def visit(node):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): return
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name): calls.add(node.func.id)
        for child in ast.iter_child_nodes(node): visit(child)

    for node in tree.body: visit(node)
    return calls

In [ ]:
#| export
def _execution_warning_function(node):
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return False
    if node.decorator_list: return False
    return node.name == "main" or not node.name.startswith("_")

In [ ]:
#| export
def _execution_function_defs(nb, up2id=None):
    defs = {}
    for idx, cell in _executed_cells(nb, up2id=up2id):
        tree = _parse_executed_code_cell(cell)
        if tree is None: continue
        for node in tree.body:
            if _execution_warning_function(node):
                defs.setdefault(node.name, {"cell_id": getattr(cell, "id", ""), "line": getattr(node, "lineno", None)})
    return defs

In [ ]:
#| export
def _uncalled_function_warnings(nb, up2id=None):
    called = set()
    for idx, cell in _executed_cells(nb, up2id=up2id):
        tree = _parse_executed_code_cell(cell)
        if tree is not None: called.update(_top_level_call_names(tree))
    warnings = []
    for name, info in sorted(_execution_function_defs(nb, up2id=up2id).items()):
        if name in called: continue
        location = f"cell id={info['cell_id']}" if info.get("cell_id") else "an executed cell"
        warnings.append(
            f"function {name!r} defined in {location} was not called by executed cells; "
            "add a focused test or example, or add a decorator if it is invoked externally."
        )
    return warnings

In [ ]:
#| export
def _print_uncalled_function_warnings(nb, up2id=None):
    warnings = _uncalled_function_warnings(nb, up2id=up2id)
    if not warnings: return []
    print("Execution warnings:")
    for warning in warnings: print(f"- {warning}")
    return warnings

### The public executor

`exec_nb` is the user-facing wrapper around the execution engine. It writes outputs back to the notebook, prints visible outputs when requested, and supports partial execution through `up2id` or `chapter`.

In [ ]:
#| export
@call_parse
@tracked_call
def exec_nb(
    path: str,  # Notebook path
    dest: str | None = None,  # Destination path; defaults to overwriting path
    exc_stop: bool = False,  # Stop on exceptions
    up2id: int | str | None = None,  # Execute first N cells, or through this cell id
    chapter: str | None = None,  # Execute through this chapter, inclusive
    timeout: int = 30,  # Per-cell timeout in seconds; <=0 disables timeouts
    show_output: bool = True,  # Print saved cell outputs and errors after execution
    verbose: bool = False,  # Show stdout/stderr live while executing
    safe: bool = True,  # Use safepyrun instead of the legacy execnb shell
    allow: str | None = None,  # Comma-separated or literal list of trusted callables to allow
    ok_dests: str | None = None,  # Comma-separated or literal list of allowed write destinations
    cache_httpx: bool = False,  # Return cached httpx responses instead of making live calls
    cache_dir: str | None = None,  # Directory containing cachy.jsonl; defaults to project root
    cache_domains: str | None = None,  # Comma-separated or literal list of cacheable domains
    allow_new: bool = False,  # Execute cells without prior user/nbskill execution approval
    check_only: bool = False,  # Run in memory without writing notebook outputs or metadata
):
    "Execute a notebook with safe Python by default and local project imports available."
    dest = None if check_only else (dest or path)
    chapter_title = None
    if chapter is not None:
        if up2id is not None: raise ValueError("Use either chapter or up2id, not both")
        with notebook_locks(path):
            nb = _read_nb(path)
            span = one_chapter(nb.cells, chapter)
        up2id, chapter_title = span["end"], span["title"]
    preproc, postproc = _exec_limiters(up2id)
    nb = _execute_nb(
        path, dest=dest, exc_stop=exc_stop, preproc=preproc, postproc=postproc,
        timeout=timeout, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests,
        cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains,
        allow_new=allow_new,
    )
    mode = "safe" if safe else "unsafe"
    target = "not written" if check_only else dest
    msg = f"Executed {path} -> {target} ({mode})"
    if check_only: msg += " (check_only=True)"
    if chapter_title is not None: msg += f" (chapter={chapter_title!r}, up2id={up2id})"
    elif up2id is not None: msg += f" (up2id={up2id})"
    if timeout and timeout > 0: msg += f" (timeout={timeout}s)"
    print(msg)
    if show_output:
        if check_only: _print_outputs_from_nb(nb, up2id=up2id)
        else: _print_nb_outputs(dest, up2id=up2id)
    _print_uncalled_function_warnings(nb, up2id=up2id)
    return cli_return(Path(path) if check_only else Path(dest))

In [ ]:
root = demo_path("03_execute_project")
try:
    root.mkdir()
    (root / "pyproject.toml").write_text("[project]\nname = 'local-demo'\n", encoding="utf-8")
    pkg = root / "local_demo"
    pkg.mkdir()
    (pkg / "__init__.py").write_text("def meaning():\n    return 42\n", encoding="utf-8")
    nbs = root / "nbs"
    nbs.mkdir()

    path = nbs / "sample.ipynb"
    write_nb(
        str(path),
        "%%code\nfrom local_demo import meaning\nprint(meaning())\nassert meaning() == 42",
        replace=True,
        export=False,
    )
    exec_nb(str(path), timeout=5, allow="local_demo.meaning")
    text = "".join(_output_text(output) for output in _read_nb(path).cells[0].outputs)
    assert "refusing to execute unapproved cell" in text
    exec_nb(str(path), timeout=5, allow="local_demo.meaning", allow_new=True)
    text = "".join(_output_text(output) for output in _read_nb(path).cells[0].outputs)
    assert "42" in text

    check = nbs / "check_only.ipynb"
    write_nb(str(check), "%%code\nprint('check only')", replace=True, export=False)
    before = check.read_text(encoding="utf-8")
    exec_nb(str(check), timeout=5, allow_new=True, check_only=True)
    after = check.read_text(encoding="utf-8")
    assert after == before

    cross = nbs / "cross.ipynb"
    write_nb(str(cross), "%%code\\nx = 2\\n---\\n%%code\\nprint(x + 3)", replace=True, export=False)
    exec_nb(str(cross), timeout=5, allow_new=True)
    text = "".join(_output_text(output) for output in _read_nb(cross).cells[1].outputs)
    assert "5" in text

    slow = nbs / "slow.ipynb"
    write_nb(str(slow), "%%code\nimport time\ntime.sleep(2)", replace=True, export=False)
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = _read_nb(slow).cells[0]
    assert cell.metadata[_TIMEOUT_HASH_KEY]
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "ran longer than 1s" in text
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = _read_nb(slow).cells[0]
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "skipped cell" in text
    update_cell(str(slow), "print('fast now')", cell_id=cell.id, export=False)
    exec_nb(str(slow), timeout=1, allow_new=True, safe=False)
    cell = _read_nb(slow).cells[0]
    assert _TIMEOUT_HASH_KEY not in cell.metadata
    text = "".join(_output_text(output) for output in cell.outputs)
    assert "fast now" in text

    blocked_cases = {
        "path_write": "from pathlib import Path\nPath('bad.txt').write_text('bad')",
        "open_write": "open('bad.txt', 'w').write('bad')",
        "remove": "import os\nos.remove('missing.txt')",
        "rmtree": "import shutil\nshutil.rmtree('missing')",
        "subprocess": "import subprocess\nsubprocess.run(['true'])",
        "magic": "%time 1 + 1",
        "shell": "!echo unsafe",
    }
    for name, source in blocked_cases.items():
        nb_path = nbs / f"{name}.ipynb"
        write_nb(str(nb_path), f"%%code\n{source}", replace=True, export=False)
        exec_nb(str(nb_path), timeout=5, allow_new=True)
        text = "".join(_output_text(output) for output in _read_nb(nb_path).cells[0].outputs)
        assert text
        if name in {"magic", "shell"}: assert "blocks IPython" in text

    allowed_write = nbs / "allowed_write.ipynb"
    ok_file = root / "allowed.txt"
    write_nb(
        str(allowed_write),
        f"%%code\nfrom pathlib import Path\nPath({str(ok_file)!r}).write_text('ok')\nprint(Path({str(ok_file)!r}).read_text())",
        replace=True,
        export=False,
    )
    exec_nb(str(allowed_write), timeout=5, ok_dests=str(root), allow_new=True)
    assert ok_file.read_text(encoding="utf-8") == "ok"

    import httpx as _httpx
    live = nbs / "live_httpx.ipynb"
    write_nb(str(live), "%%code\nimport httpx\nhttpx.get('https://api.openai.com/v1/test')", replace=True, export=False)
    exec_nb(str(live), timeout=5, allow_new=True)
    text = "".join(_output_text(output) for output in _read_nb(live).cells[0].outputs)
    assert "blocked live httpx call" in text

    cached = nbs / "cached_httpx.ipynb"
    url = "https://api.openai.com/v1/test"
    request = _httpx.Request("GET", url)
    key = _cachy_key(request)
    (root / "cachy.jsonl").write_text(
        json.dumps({"key": key, "response": "cached body", "headers": {"content-type": "text/plain"}, "status_code": 200}) + "\n",
        encoding="utf-8",
    )
    write_nb(str(cached), f"%%code\nimport httpx\nr = httpx.get({url!r})\nprint(r.text)", replace=True, export=False)
    exec_nb(str(cached), timeout=5, cache_httpx=True, cache_dir=str(root), allow_new=True)
    text = "".join(_output_text(output) for output in _read_nb(cached).cells[0].outputs)
    assert "cached body" in text

    miss = nbs / "miss_httpx.ipynb"
    miss_url = "https://api.openai.com/v1/miss"
    write_nb(str(miss), f"%%code\nimport httpx\nhttpx.get({miss_url!r})", replace=True, export=False)
    exec_nb(str(miss), timeout=5, cache_httpx=True, cache_dir=str(root), allow_new=True)
    text = "".join(_output_text(output) for output in _read_nb(miss).cells[0].outputs)
    assert "no cached httpx response" in text
finally:
    remove_demo_path(root)

In [ ]:
warning_root = demo_path("03_execute_warnings")
try:
    warning_root.mkdir()
    warning_nbs = warning_root / "nbs"
    warning_nbs.mkdir()

    uncalled = warning_nbs / "uncalled.ipynb"
    write_nb(
        str(uncalled),
        "%%code\ndef unused_demo():\n    return 1\n---\n%%code\nprint('done')",
        replace=True,
        export=False,
    )
    out = _StringIO()
    with _redirect_stdout(out):
        exec_nb(str(uncalled), timeout=5, allow_new=True, show_output=False)
    warning_text = out.getvalue()
    assert "Execution warnings:" in warning_text
    assert "unused_demo" in warning_text

    called = warning_nbs / "called.ipynb"
    write_nb(
        str(called),
        "%%code\ndef used_demo():\n    return 1\n---\n%%code\nused_demo()",
        replace=True,
        export=False,
    )
    out = _StringIO()
    with _redirect_stdout(out):
        exec_nb(str(called), timeout=5, allow_new=True, show_output=False)
    assert "used_demo" not in out.getvalue()

    decorated = warning_nbs / "decorated.ipynb"
    write_nb(
        str(decorated),
        "%%code\ndef _route(func):\n    return func\n\n@_route\ndef api_handler():\n    return 1",
        replace=True,
        export=False,
    )
    out = _StringIO()
    with _redirect_stdout(out):
        exec_nb(str(decorated), timeout=5, allow_new=True, show_output=False)
    assert "api_handler" not in out.getvalue()
finally:
    remove_demo_path(warning_root)

### Testing after edits

The write tools can ask for a notebook test immediately after changing a notebook. These helpers summarize saved error outputs so a failed write/test cycle reports the useful cell-level problem.

In [ ]:
#| export
def _notebook_error_summaries(path, up2id=None):
    with notebook_locks(path):
        nb = _read_nb(path)
        errors = []
        for idx, cell in _executed_cells(nb, up2id=up2id):
            for output in cell.get("outputs", []):
                if output.get("output_type") == "error":
                    ename = output.get("ename", "Error")
                    evalue = output.get("evalue", "")
                    errors.append(f"id={cell.id} {ename}: {evalue}".strip())
        return errors

In [ ]:
#| export
def run_notebook_test(path, timeout=30):
    print(f"Running notebook test with safe execution on {path} (timeout={timeout}s)")
    _execute_nb(path, dest=path, exc_stop=False, timeout=timeout, verbose=False)
    _print_nb_outputs(path)
    errors = _notebook_error_summaries(path)
    if errors:
        sys.stdout.flush()
        cli_error("Notebook test failed after writing/execution: " + "; ".join(errors))